# Harpy SpatialData Vectorization + Aggregation Test (`SLIDE-0329` written sdata)

This notebook is a follow-on to `spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.ipynb`.

It keeps the same written image-only `SpatialData` store as the starting point, then:

1. reads the written `.sdata.zarr`
2. attaches the existing full-slide whole-cell and nuclear mask TIFFs as labels
3. vectorizes those labels with Harpy
4. aggregates channel intensities over the label layers with Harpy
5. checks that mask IDs, vectorized shapes, and aggregate table IDs still agree

This intentionally skips InstanSeg so we can isolate vectorization and aggregation on the written full-slide store that now loads correctly.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from time import perf_counter

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import dask.array as da
import numpy as np
import pandas as pd
import spatialdata as sd
import tifffile
from dask.distributed import Client, LocalCluster
from spatialdata import SpatialData, read_zarr
from spatialdata.models import Labels2DModel
from spatialdata.transformations import Identity, set_transformation

import harpy as hp

print("Python:", sys.version.split()[0])
print("spatialdata:", getattr(sd, "__version__", "unknown"))
print("harpy:", getattr(hp, "__version__", "unknown"))
print("tifffile:", getattr(tifffile, "__version__", "unknown"))
print("pandas:", getattr(pd, "__version__", "unknown"))
print("numpy:", getattr(np, "__version__", "unknown"))


In [ ]:
# User parameters
OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
SOURCE_SDATA_PATH = OUTPUTS_DIR / "spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr"
CHANNEL_MAP_PATH = OUTPUTS_DIR / "channel_map.generated.json"
CELL_MASK_PATH = OUTPUTS_DIR / "masks" / "SLIDE-0329_whole_cell.tiff"
NUCLEAR_MASK_PATH = OUTPUTS_DIR / "masks" / "SLIDE-0329_nuclear.tiff"

WORKDIR = OUTPUTS_DIR / "harpy_vectorize_aggregate_written_sdata"
WORKDIR.mkdir(parents=True, exist_ok=True)
FINAL_ZARR_PATH = WORKDIR / "SLIDE-0329_vectorize_aggregate_test.sdata.zarr"

IMAGE_LAYER = "full_image"
CELL_LABELS = "cell_labels"
NUC_LABELS = "nuclear_labels"
CELL_SHAPES = "cell_boundaries"
NUC_SHAPES = "nuclear_boundaries"
CELL_TABLE = "agg_cell_labels"
NUC_TABLE = "agg_nuclear_labels"

MASK_CHUNKS = (4096, 4096)
AGGREGATION_CHUNKS = 1000
ALLOCATE_MODE = "sum"
ALLOCATE_OBS_STATS = ["var", "skew", "count"]
CELL_INSTANCE_SIZE_KEY = "cell_size"
NUC_INSTANCE_SIZE_KEY = "nucleus_size"
WRITE_OUTPUT = False
OPEN_IN_NAPARI = False

required_paths = [SOURCE_SDATA_PATH, CHANNEL_MAP_PATH, CELL_MASK_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

print({
    "source_sdata": str(SOURCE_SDATA_PATH),
    "channel_map": str(CHANNEL_MAP_PATH),
    "cell_mask": str(CELL_MASK_PATH),
    "nuclear_mask": str(NUCLEAR_MASK_PATH),
    "workdir": str(WORKDIR),
    "final_zarr": str(FINAL_ZARR_PATH),
    "mask_chunks": MASK_CHUNKS,
    "aggregation_chunks": AGGREGATION_CHUNKS,
    "allocate_mode": ALLOCATE_MODE,
    "allocate_obs_stats": ALLOCATE_OBS_STATS,
    "write_output": WRITE_OUTPUT,
})


## Dask client

Use a small local cluster so vectorization and aggregation follow the same Harpy-friendly pattern as the other notebook.


In [ ]:
cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=1,
    memory_limit="20GB",
)
client = Client(cluster)
print(client.dashboard_link)
client


## Read the written SpatialData store

This is the image-only store written by `spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.ipynb`.


In [ ]:
source_sdata = read_zarr(SOURCE_SDATA_PATH)
img = hp.im.get_dataarray(source_sdata, IMAGE_LAYER)

print({
    "source_path": str(SOURCE_SDATA_PATH),
    "is_backed": source_sdata.is_backed(),
    "images": list(source_sdata.images.keys()),
    "labels": list(source_sdata.labels.keys()),
    "shapes": list(source_sdata.shapes.keys()),
    "tables": list(source_sdata.tables.keys()),
    "image_layer": IMAGE_LAYER,
    "img_dims": tuple(img.dims),
    "img_shape": tuple(int(v) for v in img.shape),
    "img_chunks": img.chunks,
    "channel_count": len(img.coords["c"].values),
    "first_channels": list(map(str, img.coords["c"].values[:10])),
})

source_sdata


In [ ]:
# Optional tiny compute sanity check
test_patch = img.data[0, :256, :256].compute()
print("test_patch shape:", test_patch.shape, "dtype:", test_patch.dtype)


## Attach existing mask TIFFs as labels

Keep the source image store untouched by building a new `SpatialData` object that reuses the written image layer and adds the mask labels on top.


In [ ]:
cell_mask = tifffile.memmap(CELL_MASK_PATH)
nuclear_mask = tifffile.memmap(NUCLEAR_MASK_PATH) if NUCLEAR_MASK_PATH.exists() else None

image_canvas = tuple(int(v) for v in img.shape[-2:])
assert tuple(cell_mask.shape) == image_canvas, (
    f"Cell mask shape {tuple(cell_mask.shape)} must match image canvas {image_canvas}"
)
if nuclear_mask is not None:
    assert tuple(nuclear_mask.shape) == image_canvas, (
        f"Nuclear mask shape {tuple(nuclear_mask.shape)} must match image canvas {image_canvas}"
    )

cell_mask_da = da.from_array(cell_mask, chunks=MASK_CHUNKS)
labels = {
    CELL_LABELS: Labels2DModel.parse(cell_mask_da, dims=("y", "x")),
}

if nuclear_mask is not None:
    nuclear_mask_da = da.from_array(nuclear_mask, chunks=MASK_CHUNKS)
    labels[NUC_LABELS] = Labels2DModel.parse(nuclear_mask_da, dims=("y", "x"))

for label in labels.values():
    set_transformation(label, Identity(), to_coordinate_system="global")

sdata = SpatialData(images={IMAGE_LAYER: source_sdata.images[IMAGE_LAYER]}, labels=labels)

def nonzero_instance_ids(mask_array):
    ids = np.unique(mask_array)
    ids = ids[ids > 0].astype(int)
    return sorted(ids.astype(str).tolist(), key=int)

cell_mask_ids = nonzero_instance_ids(cell_mask)
nuclear_mask_ids = nonzero_instance_ids(nuclear_mask) if nuclear_mask is not None else []

print({
    "labels": list(sdata.labels.keys()),
    "cell_mask_shape": tuple(cell_mask.shape),
    "cell_mask_dtype": str(cell_mask.dtype),
    "cell_label_count": len(cell_mask_ids),
    "nuclear_mask_shape": tuple(nuclear_mask.shape) if nuclear_mask is not None else None,
    "nuclear_mask_dtype": str(nuclear_mask.dtype) if nuclear_mask is not None else None,
    "nuclear_label_count": len(nuclear_mask_ids),
})

sdata


## Vectorize labels with Harpy

This is the main raster-to-vector test against the written full-slide image store.


In [ ]:
vectorization_stats = {}

for labels_layer, output_layer in [
    (CELL_LABELS, CELL_SHAPES),
    (NUC_LABELS, NUC_SHAPES),
]:
    if labels_layer not in sdata.labels:
        continue
    print(f"[vectorize] {labels_layer} -> {output_layer}")
    started = perf_counter()
    sdata = hp.sh.vectorize(
        sdata,
        labels_layer=labels_layer,
        output_layer=output_layer,
        overwrite=True,
    )
    elapsed = perf_counter() - started
    shape_df = sdata.shapes[output_layer]
    vectorization_stats[output_layer] = {
        "rows": int(len(shape_df.index)),
        "columns": list(shape_df.columns),
        "seconds": round(elapsed, 2),
    }
    print(vectorization_stats[output_layer])

print("shapes:", list(sdata.shapes.keys()))
vectorization_stats


In [ ]:
if CELL_SHAPES in sdata.shapes:
    display(sdata.shapes[CELL_SHAPES].head())
if NUC_SHAPES in sdata.shapes:
    display(sdata.shapes[NUC_SHAPES].head())


## Aggregate image intensities over the label layers

Keep this close to the Harpy tutorial flow and use `allocate_intensity(...)` on the raster label layers.


In [ ]:
aggregate_stats = {}

for labels_layer, table_layer, instance_size_key in [
    (CELL_LABELS, CELL_TABLE, CELL_INSTANCE_SIZE_KEY),
    (NUC_LABELS, NUC_TABLE, NUC_INSTANCE_SIZE_KEY),
]:
    if labels_layer not in sdata.labels:
        continue
    print(f"[aggregate] {labels_layer} -> {table_layer}")
    started = perf_counter()
    sdata = hp.tb.allocate_intensity(
        sdata,
        img_layer=IMAGE_LAYER,
        labels_layer=labels_layer,
        output_layer=table_layer,
        mode=ALLOCATE_MODE,
        obs_stats=ALLOCATE_OBS_STATS,
        instance_size_key=instance_size_key,
        chunks=AGGREGATION_CHUNKS,
        append=False,
        calculate_center_of_mass=True,
        overwrite=True,
    )
    elapsed = perf_counter() - started
    table = sdata.tables[table_layer]
    aggregate_stats[table_layer] = {
        "shape": tuple(table.shape),
        "obs_columns": list(table.obs.columns),
        "first_features": table.var_names[:8].tolist(),
        "seconds": round(elapsed, 2),
    }
    print(aggregate_stats[table_layer])

print("tables:", list(sdata.tables.keys()))
aggregate_stats


In [ ]:
if CELL_TABLE in sdata.tables:
    display(sdata.tables[CELL_TABLE].to_df().head())
    display(sdata.tables[CELL_TABLE].obs.head())
if NUC_TABLE in sdata.tables:
    display(sdata.tables[NUC_TABLE].to_df().head())
    display(sdata.tables[NUC_TABLE].obs.head())


## Compare IDs across masks, shapes, and tables

This is a quick integrity check so we can see whether vectorization and aggregation preserved the instance IDs from the original masks.


In [ ]:
def table_instance_ids(adata):
    if "instance_id" in adata.obs.columns:
        values = adata.obs["instance_id"].astype(str).tolist()
    else:
        values = adata.obs_names.astype(str).tolist()
    return sorted(values, key=int)

def shape_instance_ids(shape_df):
    if "label" in shape_df.columns:
        values = shape_df["label"].astype(int).astype(str).tolist()
    elif "instance_id" in shape_df.columns:
        values = shape_df["instance_id"].astype(str).tolist()
    else:
        values = list(map(str, shape_df.index.tolist()))
    return sorted(values, key=int)

comparison = {
    CELL_LABELS: {
        "mask_ids": cell_mask_ids,
        "shape_ids": shape_instance_ids(sdata.shapes[CELL_SHAPES]) if CELL_SHAPES in sdata.shapes else [],
        "table_ids": table_instance_ids(sdata.tables[CELL_TABLE]) if CELL_TABLE in sdata.tables else [],
    },
}
if NUC_LABELS in sdata.labels:
    comparison[NUC_LABELS] = {
        "mask_ids": nuclear_mask_ids,
        "shape_ids": shape_instance_ids(sdata.shapes[NUC_SHAPES]) if NUC_SHAPES in sdata.shapes else [],
        "table_ids": table_instance_ids(sdata.tables[NUC_TABLE]) if NUC_TABLE in sdata.tables else [],
    }

summary_rows = []
for layer_name, payload in comparison.items():
    mask_id_set = set(payload["mask_ids"])
    shape_id_set = set(payload["shape_ids"])
    table_id_set = set(payload["table_ids"])
    summary_rows.append({
        "layer": layer_name,
        "mask_count": len(mask_id_set),
        "shape_count": len(shape_id_set),
        "table_count": len(table_id_set),
        "mask_eq_shape": mask_id_set == shape_id_set,
        "mask_eq_table": mask_id_set == table_id_set,
        "shape_eq_table": shape_id_set == table_id_set,
        "mask_minus_shape": sorted(mask_id_set - shape_id_set, key=int)[:10],
        "shape_minus_mask": sorted(shape_id_set - mask_id_set, key=int)[:10],
        "mask_minus_table": sorted(mask_id_set - table_id_set, key=int)[:10],
        "table_minus_mask": sorted(table_id_set - mask_id_set, key=int)[:10],
    })

comparison_df = pd.DataFrame(summary_rows)
display(comparison_df)
comparison_df


## Optional visualization


In [ ]:
if OPEN_IN_NAPARI:
    from napari_spatialdata import Interactive
    Interactive(sdata)
else:
    print("Set OPEN_IN_NAPARI = True to inspect interactively.")


## Optional save

Leave `WRITE_OUTPUT = False` by default so this notebook can probe the behavior without overwriting the original working image-only store.


In [ ]:
if WRITE_OUTPUT:
    if FINAL_ZARR_PATH.exists():
        import shutil
        shutil.rmtree(FINAL_ZARR_PATH)
    sdata.write(FINAL_ZARR_PATH, overwrite=True)
    reloaded = sd.read_zarr(FINAL_ZARR_PATH)
    print("Wrote:", FINAL_ZARR_PATH)
    print("images:", list(reloaded.images.keys()))
    print("labels:", list(reloaded.labels.keys()))
    print("shapes:", list(reloaded.shapes.keys()))
    print("tables:", list(reloaded.tables.keys()))
    reloaded
else:
    print("WRITE_OUTPUT is False. No zarr written.")


## Close the Dask client


In [ ]:
client.close()
